# MediBill-Env — SFT Quickstart on Free Colab T4

**Before you click anything:**
1. Top menu → **Runtime → Change runtime type → T4 GPU → Save**.
2. Then **Runtime → Run all**.
3. The Drive-mount cell will pause once for you to paste an auth code. After that, it runs unattended for ~75 minutes.

Total wallclock: ~80 min. Adapter + log are saved to `/content/drive/MyDrive/medibill/` so a free-tier disconnect does not lose your work.

**What this notebook does NOT do:** edit the pitch, edit the env, or change any committed code. It only trains an SFT adapter and writes the eval table to a log file.

## 1. Confirm GPU is attached

In [ ]:
!nvidia-smi

Expect a table containing `Tesla T4`. If you see `command not found` or no GPU, redo the runtime change above.

## 2. Mount Google Drive (so adapter survives free-tier disconnects)

When the popup appears, allow access and paste the code back into the input box.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/medibill/adapters /content/drive/MyDrive/medibill/runs

## 3. Clone the repo

In [ ]:
!git clone https://github.com/Algoace1403/METAHackthon2026 /content/METAHackthon2026
%cd /content/METAHackthon2026

## 4. Install dependencies (~3 min)

Pip warnings about pre-installed packages are normal. Watch for **red** errors only.

In [ ]:
!pip install -e '.[train]' -q

In [ ]:
!pip install -q 'unsloth[cu121] @ git+https://github.com/unslothai/unsloth.git'

## 5. Smoke test (~60 seconds)

Catches packaging/path/GPU issues cheaply before the 75-minute training run.

**If the next cell exits non-zero, STOP.** Paste the last 20 lines back into your chat and I will diagnose. Do not run cell 6.

In [ ]:
!python -m medibill.colab_smoke

## 6. Train + evaluate (~75 minutes unattended)

Training prints one line every 5 optimizer steps. After training, an automatic per-task eval runs and prints a table like:

```
easy_cashless          n=4  sft_adapter=0.??  scripted=1.000  Δ=±0.???
medium_multi_payer     n=4  sft_adapter=0.??  scripted=1.000  Δ=±0.???
hard_drift             n=4  sft_adapter=0.??  scripted=0.754  Δ=±0.???
```

Walk away. Do not click anything in this notebook for ~75 minutes. Keep the Colab tab open and the Mac awake to avoid the free-tier idle disconnect.

In [ ]:
!python -m medibill.sft_colab \
    --dataset datasets/sft_v1.jsonl \
    --eval traces/eval.jsonl \
    --out /content/drive/MyDrive/medibill/adapters/sft_v1/ \
    --batch-size 2 --grad-accum 8 \
    2>&1 | tee /content/drive/MyDrive/medibill/runs/sft_v1_stdout.log

## 7. Inspect the eval table

Run this cell when training is done. It prints the bottom of the log so you can copy the per-task table into your pitch deck (slide 5 4th-bar) or paste back to chat for review.

In [ ]:
!tail -40 /content/drive/MyDrive/medibill/runs/sft_v1_stdout.log

## What to do with the result

- **If `sft_adapter` on `hard_drift` > scripted (0.76)** → you have a 4th bar story. Paste the table to chat; I will update the pitch.
- **If `sft_adapter` ≤ scripted within ±0.03** → structural ceiling reached as documented in `docs/round2-spec-v3.md §7.6`. Pitch stays env-first; we mention SFT pipeline ran but did not surpass scripted.
- **If training crashed** → paste the last 30 lines of the log. The pitch is unaffected.

The adapter is at `/content/drive/MyDrive/medibill/adapters/sft_v1/` and persists across notebook sessions.